# 04 — Feature Engineering & Leakage Check

Phase 6 (build predictive features) and Phase 7 (prove none of them could only be known *after* the fraud decision is made).

In [1]:
import sys
from pathlib import Path
import pandas as pd

sys.path.insert(0, str(Path('..').resolve() / 'src'))
import features

df = pd.read_csv('../data/processed/transactions_clean.csv')
df.shape

(283726, 32)

## Phase 6 — Features created

In [2]:
df_fe = features.engineer_features(df)
new_cols = [c for c in df_fe.columns if c not in df.columns]
print('New columns:', new_cols)
df_fe[new_cols].describe()

New columns: ['log_amount', 'hour_of_day', 'elapsed_day', 'transactions_per_hour_bucket']


,log_amount,hour_of_day,elapsed_day,transactions_per_hour_bucket
count,283726.000000,283726.000000,283726.000000,283726.000000
mean,3.153760,14.045646,0.491636,14474.357436
std,1.657080,5.834817,0.499931,3911.898697
min,0.000000,0.000000,0.000000,2204.000000
25%,1.887070,10.000000,0.000000,15323.000000
50%,3.135494,15.000000,0.000000,16130.000000
75%,4.363226,19.000000,1.000000,16548.000000
max,10.153941,23.000000,1.000000,17629.000000


| Feature | Built from | What it captures |
|---|---|---|
| `log_amount` | `Amount` | Variance-stabilizing transform — `Amount` is heavily right-skewed (03_eda.ipynb §2) |
| `hour_of_day` | `Time` | Time-of-day pattern (03_eda.ipynb §4/§7 found this is real, not noise) |
| `elapsed_day` | `Time` | Which of the two observed days (0 or 1) |
| `transactions_per_hour_bucket` | `Time` (via `hour_of_day`) | Volume/context signal — is this an unusually busy or quiet hour |

### Features deliberately NOT created

- **`transactions_per_customer`, `amount_vs_customer_average`** — this dataset has no customer or card identifier at all (data/README.md). There is no honest way to build a per-customer feature without a customer key; fabricating one (e.g. clustering on `V1`-`V28` and calling the cluster a "customer") would be inventing information the data does not contain, not engineering a feature.
- **`day_of_week`** — `Time` is seconds since the first transaction in the dataset, not a real timestamp, so there's no genuine calendar date underneath it. `elapsed_day` (above) is the honest version: which of the two observed days, nothing more.

## Phase 7 — Data leakage check

The test for every candidate feature: **could this realistically be known at the moment a fraud decision is made on this transaction, before any investigation outcome exists?** If the answer is no, it's excluded — no exceptions, regardless of how predictive it looks.

In [3]:
leakage_review = pd.DataFrame([
    {'feature': 'V1-V28', 'available_at_decision_time': True,
     'reasoning': 'PCA components of the ORIGINAL pre-transaction features '
                  '(merchant/location/device-type signals per the dataset ' 
                  "documentation) - computed independently of the fraud "
                  'outcome. Legitimate pre-transaction inputs.'},
    {'feature': 'Amount / log_amount', 'available_at_decision_time': True,
     'reasoning': 'Known the instant a transaction is initiated.'},
    {'feature': 'hour_of_day / elapsed_day', 'available_at_decision_time': True,
     'reasoning': 'Derived purely from Time, known at transaction time.'},
    {'feature': 'transactions_per_hour_bucket', 'available_at_decision_time': True,
     'reasoning': 'Computed only from Time; does not reference is_fraud for any row. '
                  'Caveat: this offline version is a whole-dataset (symmetric) count - '
                  'a live system could only count PAST transactions in the current hour, '
                  'so this is a research-pipeline simplification, flagged not hidden.'},
    {'feature': 'is_fraud (target)', 'available_at_decision_time': False,
     'reasoning': 'This IS the thing being predicted - obviously excluded from X.'},
])
leakage_review

,feature,available_at_decision_time,reasoning
0,V1-V28,True,PCA components of the ORIGINAL pre-transaction...
1,Amount / log_amount,True,Known the instant a transaction is initiated.
2,hour_of_day / elapsed_day,True,"Derived purely from Time, known at transaction..."
3,transactions_per_hour_bucket,True,Computed only from Time; does not reference is...
4,is_fraud (target),False,This IS the thing being predicted - obviously ...


**No investigation-outcome-style column exists in this dataset** (e.g. there is no `chargeback_confirmed`, `investigation_result`, or `resolved_date` field to accidentally include) — unlike a richer production feed, so the leakage risk here is more subtle than "drop the obviously circular column". The two real leakage risks in *this* project were:

1. **Duplicate rows spanning train/test** — handled in `02_data_quality.ipynb` by dropping exact duplicates before splitting (Phase 3), so no identical row can appear in both train and test.
2. **Oversampling before splitting** — addressed procedurally in `05_model_training.ipynb`: any resampling (SMOTE) is applied strictly after the train/validation/test split, and only to the training fold.

Every feature that survives to `src/features.MODEL_FEATURE_COLUMNS` passes the leakage test above.

In [4]:
import features as _f
_f.MODEL_FEATURE_COLUMNS

['V1',
 'V2',
 'V3',
 'V4',
 'V5',
 'V6',
 'V7',
 'V8',
 'V9',
 'V10',
 'V11',
 'V12',
 'V13',
 'V14',
 'V15',
 'V16',
 'V17',
 'V18',
 'V19',
 'V20',
 'V21',
 'V22',
 'V23',
 'V24',
 'V25',
 'V26',
 'V27',
 'V28',
 'Amount',
 'log_amount',
 'hour_of_day',
 'elapsed_day',
 'transactions_per_hour_bucket']

In [5]:
out_dir = Path('..') / 'data' / 'processed'
out_dir.mkdir(parents=True, exist_ok=True)
df_fe.to_csv(out_dir / 'transactions_features.csv', index=False)
print('saved', out_dir / 'transactions_features.csv', df_fe.shape)

saved ..\data\processed\transactions_features.csv (283726, 36)
